# The Bison Shuttle Feed: a pipeline that refuses to be wrong

**MATH 153 · Data Wrangling / Data Engineering** — in-class activity, one session
Howard University · Department of Mathematics
Moussa Doumbia, Ph.D.

---

## The problem

Transportation Services publishes a **daily ridership report**: how many people boarded at each stop.
The numbers come from the automatic passenger counters (APC) on the shuttles, exported as one CSV per
service day — three routes, thirty runs, about 124 stop-events.

The counters are older than the contract. When one drops out it does not leave the field blank:
**it writes a number.** A boarding count of `-1`. A position of `0.0, 0.0`. A vehicle called
`UNKNOWN`. These are **sentinel values** — real-looking data standing in for *"I don't know"*.

You have three service days. **One is ordinary. Two are not**, and each is broken in a different way.
Your pipeline is not told which.

> **The whole activity in one sentence:** a sentinel is a lie told in the correct data type,
> and the type system will never catch it for you.

*Everything in this dataset is invented. Real ridership and vehicle telemetry would not belong in a
teaching repository.*

## How the hour runs

| Minutes | What happens |
|---|---|
| 0–8 | Part 1 together: read a file carelessly and watch it lie to you |
| 8–15 | Part 2 and 3 together: the catalogue, and a pipeline with **no gates** |
| 15–40 | **Part 4 — you write seven gates.** One assertion each, under four lines |
| 40–48 | Part 5 and 6: run the three batches, write down which gate fired and why |
| 48–50 | Part 7 if you got there |

**Ground rules.** Relative paths only — `data/boardings_2027-03-02.csv`, never a path starting at your
home folder. Never edit a CSV by hand; every correction happens in code, with a comment. Every
assertion carries a message naming the batch, the column and the magnitude.

## Setup

In [2]:
import numpy as np
import pandas as pd
from pathlib import Path

DATA = Path("data")
pd.set_option("display.width", 110)

sorted(p.name for p in DATA.glob("*.csv"))

['boardings_2027-03-02.csv',
 'boardings_2027-03-03.csv',
 'boardings_2027-03-04.csv',
 'stops.csv',
 'vehicles.csv']

---
# Part 1. Read the file carelessly, and watch what happens

One line, the way everyone starts.

In [3]:
careless = pd.read_csv(DATA / "boardings_2027-03-02.csv")

print("boardings dtype :", careless["boardings"].dtype)
print("boardings nulls :", int(careless["boardings"].isna().sum()))
print("boardings min   :", careless["boardings"].min())
print("boardings mean  :", round(careless["boardings"].mean(), 2))
print("boardings total :", int(careless["boardings"].sum()))

boardings dtype : float64
boardings nulls : 2
boardings min   : -1.0
boardings mean  : 5.2
boardings total : 635


Two separate failures in five lines of output, and neither raised a warning.

1. **The text sentinels vanished.** `pd.read_csv` has a built-in `na_values` list — `"N/A"`, `"NULL"`,
   `"NA"`, `""` and more — and it converted them to `NaN` on the way in. Two values disappeared before
   anyone looked at the file, and we can no longer tell a field the export left blank from one it
   marked `N/A`.
2. **The numeric sentinels survived, as numbers.** `-1` is a perfectly good integer, so it sailed into
   the mean. The minimum of **−1 boardings** is the only visible symptom, and only because we printed it.

A mean of **5.2 boardings per stop** is entirely plausible for a campus shuttle. It is wrong, and
nothing on that screen says so.

In [4]:
honest = pd.read_csv(DATA / "boardings_2027-03-02.csv", dtype=str, keep_default_na=False)

TEXT_SENTINELS = {"N/A", "NA", "n/a", "NULL", "null", "None", "-", "", "NaN"}
clean = pd.to_numeric(
    honest["boardings"].where(~honest["boardings"].str.strip().isin(TEXT_SENTINELS | {"-1"})),
    errors="coerce")

print("careless mean :", round(careless["boardings"].mean(), 2))
print("honest mean   :", round(clean.mean(), 2),
      f"  ({(clean.mean() - careless['boardings'].mean()) / careless['boardings'].mean():+.1%})")
print("careless total:", int(careless["boardings"].sum()), " honest total:", int(clean.sum()))

careless mean : 5.2
honest mean   : 5.47   (+5.1%)
careless total: 635  honest total: 640


Five percent. Not a catastrophe — which is exactly what makes it dangerous. A report that is wrong by
half gets questioned. A report that is wrong by five percent gets **believed, printed, and budgeted
against.**

Now the same sentinel in a column where it is impossible to miss.

In [5]:
print("careless fleet centre :",
      round(careless["lat"].mean(), 4), ",", round(careless["lon"].mean(), 4))

lat_h = pd.to_numeric(honest["lat"].where(honest["lat"] != "0.0")).mean()
print("honest fleet centre   :", round(lat_h, 4), ", -77.0200")
print("rows reporting 0.0, 0.0:", int((careless["lat"] == 0.0).sum()))

careless fleet centre : 37.9823 , -75.1582
honest fleet centre   : 38.924 , -77.0200
rows reporting 0.0, 0.0: 3


Three rows claim the shuttle was at latitude 0, longitude 0 — a patch of the Atlantic 5,500 miles from
Georgia Avenue. Averaged in, they drag the fleet's centre of operations **120 miles out to sea, off the
Virginia coast.**

The sentinel did not change. Only the *representation* changed. On a map it is unmissable; in a mean it
is a 2.4% shift nobody would question.

> **Keep this.** A sentinel is obvious in the representation that suits it and invisible in every other
> one. You cannot rely on noticing. You have to **check**.

---
# Part 2. The catalogue, and its backstop

A **sentinel catalogue** is written from the vendor's documentation, not from what you happen to
observe. Observing is how you find the ones the documentation forgot — which is exactly what happens
on one of these three days.

In [6]:
EXPECTED_COLUMNS = ["trip_id", "service_date", "route_id", "run_no", "stop_seq",
                    "stop_id", "arrive_time", "vehicle_id", "boardings",
                    "alightings", "onboard_after", "lat", "lon"]

COUNTS = ["boardings", "alightings", "onboard_after"]
COORDS = ["lat", "lon"]

# Per Transportation Services' APC integration note, section 3 "Quality codes".
NUMERIC_SENTINELS = {
    "boardings":     {"-1": "counter offline"},
    "alightings":    {"-1": "counter offline"},
    "onboard_after": {"-1": "counter offline"},
    "lat":           {"0.0": "no GPS fix"},
    "lon":           {"0.0": "no GPS fix"},
}
ID_SENTINELS = {"UNKNOWN", "UNK", "-1"}
TS_SENTINELS = {"1900-01-01 00:00:00", "1970-01-01 00:00:00"}

print("declared numeric sentinels:", sum(len(v) for v in NUMERIC_SENTINELS.values()))

declared numeric sentinels: 5


A catalogue is a **denylist**: it only ever knows about yesterday's sentinels. The defence against
tomorrow's is an **allowlist** — a statement of what the column is physically capable of. Nobody needs
a vendor manual to know that 255 people did not board a 52-seat shuttle at one stop.

In [7]:
# What is physically possible, independent of any vendor convention.
PHYSICAL_RANGE = {
    "boardings":     (0, 60),          # the largest vehicle in the fleet holds 52
    "alightings":    (0, 60),
    "onboard_after": (0, 60),
    "lat":           (38.80, 39.05),   # a box around the District
    "lon":           (-77.20, -76.90),
}

MISSING_TOLERANCE = 0.08    # above this share of missing boardings, the batch is not usable

pd.read_csv(DATA / "vehicles.csv")

,vehicle_id,model,seats,max_load
0,BUS-11,Gillig Low Floor 29,24,40
1,BUS-12,Gillig Low Floor 29,24,40
2,BUS-13,Gillig Low Floor 35,32,52
3,BUS-14,Gillig Low Floor 35,32,52
4,BUS-15,Ford Transit 350,14,20
5,BUS-16,Gillig Low Floor 35,32,52


---
# Part 3. The pipeline, with no gates at all

Seven stages, written out plainly. Read them — you are about to add the gates.

```
   raw CSV
      |
  [1] read        -- as text, nothing interpreted
      |
  [2] decode      -- sentinel values  ->  NaN
      |
  [3] coerce      -- text  ->  number / datetime
      |
  [4] bound       -- physically possible?
      |
  [5] join        -- attach the stop reference table
      |
  [6] rules       -- do the numbers make SENSE?
      |
  [7] report      -- boardings per stop
```

In [8]:
def stage_read(path):
    return pd.read_csv(path, dtype=str, keep_default_na=False)


def stage_decode(df, batch):
    out, log = df.copy(), []
    for col in COUNTS + COORDS:
        bad = TEXT_SENTINELS | set(NUMERIC_SENTINELS.get(col, {}))
        before = int(out[col].notna().sum())
        out[col] = out[col].where(~out[col].str.strip().isin(bad), other=pd.NA)
        log.append({"column": col, "n_decoded": before - int(out[col].notna().sum())})

    for col, bad in [("vehicle_id", ID_SENTINELS | TEXT_SENTINELS),
                     ("arrive_time", TS_SENTINELS)]:
        before = int(out[col].notna().sum())
        out[col] = out[col].where(~out[col].str.strip().isin(bad), other=pd.NA)
        log.append({"column": col, "n_decoded": before - int(out[col].notna().sum())})

    return out, pd.DataFrame(log)


def stage_coerce(df, batch):
    out = df.copy()
    for col in COUNTS + COORDS + ["run_no", "stop_seq", "stop_id"]:
        out[col] = pd.to_numeric(out[col], errors="coerce")
    out["arrive_time"] = pd.to_datetime(out["arrive_time"], errors="coerce")
    return out


def stage_bound(df, batch):
    return df


def stage_join(df, stops, batch):
    return df.merge(stops, on="stop_id", how="left", suffixes=("", "_ref"))


def stage_rules(df, batch):
    g = df.sort_values(["route_id", "run_no", "stop_seq"]).copy()
    g["prev_onboard"] = g.groupby(["route_id", "run_no"])["onboard_after"].shift(1)
    g["prev_onboard"] = g["prev_onboard"].where(g["stop_seq"] != 1, 0.0)
    g["balance"] = g["prev_onboard"] + g["boardings"] - g["alightings"]
    return g


def stage_report(df, batch):
    usable = df[df["boardings"].notna()]
    report = (usable
              .groupby(["stop_id", "stop_name", "route_id"], as_index=False)
              .agg(boardings=("boardings", "sum"), stops_served=("trip_id", "count"))
              .sort_values("boardings", ascending=False))
    return report, len(df) - len(usable)

print("seven stages defined, zero gates")

seven stages defined, zero gates


### The runner

One bad batch must not take down the job for the other two. The runner catches `AssertionError`,
records **which gate fired**, and moves on. The assertion states the rule; the runner decides what
happens when the rule is broken.

In [9]:
STOPS = (pd.read_csv(DATA / "stops.csv")
           .rename(columns={"route_id": "route_id_ref", "stop_seq": "stop_seq_ref"}))

BATCHES = ["2027-03-02", "2027-03-03", "2027-03-04"]


def run_batch(day):
    path = DATA / f"boardings_{day}.csv"
    batch = path.name
    try:
        df = stage_read(path)
        df, log = stage_decode(df, batch)
        df = stage_coerce(df, batch)
        df = stage_bound(df, batch)
        df = stage_join(df, STOPS, batch)
        df = stage_rules(df, batch)
        report, quarantined = stage_report(df, batch)
        return {"batch": batch, "status": "ok",
                "detail": f"{quarantined} rows quarantined", "report": report}
    except AssertionError as e:
        return {"batch": batch, "status": "STOPPED", "detail": str(e), "report": None}


def run_all():
    for day in BATCHES:
        r = run_batch(day)
        print(f"{r['batch']}  ->  {r['status']}")
        print(f"    {r['detail']}\n")


run_all()

boardings_2027-03-02.csv  ->  ok
    7 rows quarantined

boardings_2027-03-03.csv  ->  ok
    7 rows quarantined

boardings_2027-03-04.csv  ->  ok
    7 rows quarantined



**Three green batches. Three reports. Two of them are wrong.**

That is the state of most data pipelines in the world, and it is the thing you are here to fix.
Look at what day two is willing to publish:

In [10]:
run_batch("2027-03-03")["report"].head(5)

,stop_id,stop_name,route_id,boardings,stops_served
2,103,Annex I,A,836.0,11
3,104,Bethune Hall,A,321.0,10
0,101,Founders Library,A,318.0,12
4,105,Tubman Quadrangle,A,255.0,12
6,202,Georgia Ave & Fairmont,B,76.0,10


A stop with a few hundred boardings in a day is unremarkable at a glance. Nothing in that table says
that six of the numbers behind it came from a counter that overflowed.

---
# Part 4. Your job: seven gates

Redefine each stage below, adding the assertion marked `TODO`. **Each one is a single `assert`,
under four lines**, and each message must name the batch, the column and the magnitude.

> `assert ok` tells the person woken at 3 a.m. nothing.
> `assert ok, f"{batch}: {n} boardings above 60 (worst {worst})"` tells them what, where and how bad.

### Gates 1–4 — stage `read`

G1 and G2 are written for you. **G2 is the one people leave out and the one that bites:** on an empty
batch every later check of the form `series.all()` returns `True`, because *all* of an empty set
satisfies any condition. The pipeline runs green and publishes nothing, loudly describing it as success.

**You write G3 and G4.**

In [11]:
def stage_read(path):
    df = pd.read_csv(path, dtype=str, keep_default_na=False)
    batch = path.name

    # G1 -- the schema we were promised
    missing = [c for c in EXPECTED_COLUMNS if c not in df.columns]
    assert not missing, f"{batch}: missing columns {missing}"

    # G2 -- an empty batch is not 'no problems found'
    assert len(df) > 0, f"{batch}: zero rows -- every later .all() is vacuously True"
    
    # TODO G3 -- about 124 rows is normal. Refuse anything outside 100..200.
    assert len(df) >= 100 and len(df) <= 200, f"{batch}: rows not between 100 and 200"

    # TODO G4 -- trip_id is the key. Refuse duplicates. (Hint: .duplicated().sum())
    assert df["trip_id"].duplicated().sum() == 0, f"{batch}: duplicate values"
    
    return df

print("stage_read redefined")

stage_read redefined


### Gates 5–6 — stage `decode`

G5 is a **self-check**: it asserts that the stage did its own job. It cannot fail because of bad data,
only because of a bug in your decoding — a column left out of the loop, a sentinel spelled differently
in the catalogue than in the file. Most gates guard against the world; this kind guards against *you*,
and it is what makes a refactor safe.

G6 is the opposite: a measurement of how badly the instruments are failing.

In [12]:
def stage_decode(df, batch):
    out, log = df.copy(), []
    for col in COUNTS + COORDS:
        bad = TEXT_SENTINELS | set(NUMERIC_SENTINELS.get(col, {}))
        before = int(out[col].notna().sum())
        out[col] = out[col].where(~out[col].str.strip().isin(bad), other=pd.NA)
        log.append({"column": col, "n_decoded": before - int(out[col].notna().sum())})

    for col, bad in [("vehicle_id", ID_SENTINELS | TEXT_SENTINELS),
                     ("arrive_time", TS_SENTINELS)]:
        before = int(out[col].notna().sum())
        out[col] = out[col].where(~out[col].str.strip().isin(bad), other=pd.NA)
        log.append({"column": col, "n_decoded": before - int(out[col].notna().sum())})

    log = pd.DataFrame(log)

    # G5 -- self-check: no declared sentinel may survive this stage
    survivors = sum(int(out[c].astype(str).str.strip().isin(NUMERIC_SENTINELS.get(c, {})).sum())
                    for c in COUNTS + COORDS)
    assert survivors == 0, f"{batch}: {survivors} declared sentinels survived decoding"

    # TODO G6 -- the share of boardings now missing must not exceed MISSING_TOLERANCE.
    #            Pull the count out of `log` and divide by len(out).
    assert log["boardings"].isna().sum() / len(out) <= MISSING_TOLERANCE, "share of boardings now missing must not exceed MISSING_TOLERANCE"

    return out, log

print("stage_decode redefined")

stage_decode redefined


### Gate 7 — stage `coerce`

`errors="coerce"` turns anything it cannot parse into `NaN` **silently**. That is convenient and it is
how a format change gets into production unnoticed. An assertion belongs next to every `coerce`:
count the `NaN` before and after, and refuse if the number grew.

In [13]:
def stage_coerce(df, batch):
    out = df.copy()
    before = {c: int(out[c].isna().sum()) for c in COUNTS + COORDS}

    for col in COUNTS + COORDS + ["run_no", "stop_seq", "stop_id"]:
        out[col] = pd.to_numeric(out[col], errors="coerce")
    out["arrive_time"] = pd.to_datetime(out["arrive_time"], errors="coerce")

    created = {c: int(out[c].isna().sum()) - before[c] for c in COUNTS + COORDS}

    # TODO G7 -- no column may have gained a NaN here. If one did, the file contains
    #            a value the catalogue has never heard of. Name the columns.
    for col in created:
        assert created[col].isna().sum() == 0, f"{col} has a NaN"
    
    return out

print("stage_coerce redefined")

stage_coerce redefined


### Gate 8 — stage `bound`

The allowlist. This is the gate that catches what the catalogue never knew about, so write it to
report *which* column, *how many* rows, and the *worst* value — that message is the whole diagnosis.

Mind the `NaN`: a comparison with `NaN` is `False`, so an unguarded `~v.between(lo, hi)` would flag
every missing value as out of range. Check `v.notna()` first.

In [14]:
def stage_bound(df, batch):
    for col, (lo, hi) in PHYSICAL_RANGE.items():
        v = df[col]
        # TODO G8 -- flag non-null values outside [lo, hi]. Refuse if there are any,
        #            naming the column, the count, the range and the worst value.
        assert v <= hi or v >= lo, f"{v} column"
        
        pass
    return df

print("stage_bound redefined")

stage_bound redefined


### Gate 9 — stage `join`

A left join is a silent data-loss machine: an unmatched key gives you `NaN`, not an error. Two things
are worth asserting — that the reference table really is one row per key (`validate="many_to_one"`
does that for you, and raises on its own), and that nothing came back unmatched.

In [14]:
def stage_join(df, stops, batch):
    out = df.merge(stops, on="stop_id", how="left",
                   suffixes=("", "_ref"), validate="many_to_one")

    # TODO G9 -- no row may be left without a stop_name. Name the offending stop_ids.
    
    

    return out

print("stage_join redefined")

stage_join redefined


### Gate 10 — stage `rules`

Every number so far has been *possible*. This gate asks whether they make **sense together**.

A shuttle's load is conserved: the people aboard when it leaves a stop are the people who were aboard
when it arrived, minus those who got off, plus those who got on.

$$\text{onboard\_after} \;=\; \text{prev\_onboard} \;+\; \text{boardings} \;-\; \text{alightings}$$

Two traps. The bus starts each run empty, so `prev_onboard` is `0` at `stop_seq == 1`, not `NaN`. And
a row whose counter was offline cannot be checked at all — compare only where every term is present.

In [15]:
def stage_rules(df, batch):
    g = df.sort_values(["route_id", "run_no", "stop_seq"]).copy()
    g["prev_onboard"] = g.groupby(["route_id", "run_no"])["onboard_after"].shift(1)
    g["prev_onboard"] = g["prev_onboard"].where(g["stop_seq"] != 1, 0.0)
    g["balance"] = g["prev_onboard"] + g["boardings"] - g["alightings"]

    checkable = g["balance"].notna() & g["onboard_after"].notna()

    # TODO G10 -- among the checkable rows, the balance must equal onboard_after.
    #             Allow a tolerance of 0.5 and report where the first failure is.

    return g

print("stage_rules redefined")

stage_rules redefined


---
# Part 5. Run the three batches again

Nothing below changes. Only the gates you just wrote.

In [16]:
run_all()

boardings_2027-03-02.csv  ->  ok
    7 rows quarantined

boardings_2027-03-03.csv  ->  ok
    7 rows quarantined

boardings_2027-03-04.csv  ->  ok
    7 rows quarantined



If your gates are right you will see **one `ok` and two `STOPPED`**, and the two messages will be
different from each other. Day one still quarantines a handful of rows — the counters that were
offline — and publishes a report built only from what it could trust.

In [17]:
r = run_batch("2027-03-02")
print(r["detail"])
r["report"].head(6)

7 rows quarantined


,stop_id,stop_name,route_id,boardings,stops_served
3,104,Bethune Hall,A,113.0,12
1,102,Blackburn Center,A,88.0,12
2,103,Annex I,A,86.0,12
7,203,West Campus Gate,B,75.0,8
5,201,Howard Plaza Towers,B,68.0,9
0,101,Founders Library,A,66.0,11


---
# Part 6. Write down what you found

Answer these in the cell below, in sentences. This is the part that gets graded.

1. **Day two.** Which gate stopped it, what value triggered it, and **why could the sentinel catalogue
   never have caught this one?**
2. **Day three.** Which gate stopped it? Every number in that file is inside its physical range and no
   sentinel appears anywhere. What does that tell you about the limits of range checking?
3. **Order.** Could any *earlier* gate have caught day three if you had written it differently?
   If yes, say which and how. If no, say why not.
4. **Day one.** Seven rows were quarantined rather than fixed. Argue for or against: the report should
   have estimated the missing boardings instead of dropping them.

*Your answers here.*

1.
2.
3.
4.

---
# Part 7. If you finish early

### 7a. The canary

Every gate above has now fired at least once, so you know they work *today*. But a rule can stop
working **silently** — a column gets renamed and the rule that referenced it quietly matches nothing;
a tolerance gets widened during an incident and never tightened again.

> A rule that has not failed in six months is indistinguishable from a rule that is asleep.

The cure is one paragraph: feed the pipeline a row you *know* is bad, and assert that it gets caught.

**Expect this to fail if you run it before writing G8.** That is the point of a canary — it is the only
cell in this notebook that is *supposed* to complain while the pipeline is still unguarded.

In [18]:
def canary():
    """Push a known-bad row through and insist that stage_bound refuses it."""
    import inspect                              # noqa: F401  (for the hint below)
    df = stage_read(DATA / "boardings_2027-03-02.csv")
    df.loc[0, "boardings"] = "255"            # impossible, and undeclared
    df, _ = stage_decode(df, "canary")
    df = stage_coerce(df, "canary")
    try:
        stage_bound(df, "canary")
    except AssertionError as e:
        return f"canary caught: {e}"
    raise AssertionError(
        "CANARY ESCAPED -- a row with 255 boardings went straight through. "
        "Either G8 is not written yet, or a Part 3 cell was re-run after Part 4. "
        "Check with: inspect.getsource(stage_bound)")


print(canary())

AssertionError: CANARY ESCAPED -- a row with 255 boardings went straight through. Either G8 is not written yet, or a Part 3 cell was re-run after Part 4. Check with: inspect.getsource(stage_bound)

### 7b. A rule you have not written yet

`vehicles.csv` gives each shuttle a `max_load`. Add a gate to `stage_rules` that refuses a batch where
`onboard_after` exceeds the **max load of the vehicle that actually ran the trip** — a tighter check
than the flat `60` in `PHYSICAL_RANGE`.

Two things to think about before you write it:

- Some rows have `vehicle_id` decoded to `NaN`. Those trips cannot be checked. Is that acceptable, or
  does it deserve a gate of its own?
- `BUS-15` holds 20. If your gate is right, does day one still pass?

In [19]:
# your gate here
vehicles = pd.read_csv(DATA / "vehicles.csv")
vehicles

,vehicle_id,model,seats,max_load
0,BUS-11,Gillig Low Floor 29,24,40
1,BUS-12,Gillig Low Floor 29,24,40
2,BUS-13,Gillig Low Floor 35,32,52
3,BUS-14,Gillig Low Floor 35,32,52
4,BUS-15,Ford Transit 350,14,20
5,BUS-16,Gillig Low Floor 35,32,52


---
## What the three days taught

| Day | Defect | Caught by | Lesson |
|---|---|---|---|
| 03-02 | ordinary sentinels | decode + quarantine | the report was never wrong; it was *smaller* |
| 03-03 | `255`, undeclared | the physical bounds | a denylist only knows yesterday's sentinels |
| 03-04 | counter reset | the conservation rule | every value can be possible and the data still false |

Three gates, three different kinds of knowledge: the **vendor's** (the catalogue), the **world's**
(what is physically possible), and the **domain's** (what must add up). You need all three, and no
amount of the first two substitutes for the third.